[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/086.%20The%20Capacitated%20Facility%20Location%20Problem%20%28CFLP%29/P86-Tier-5_executed.ipynb)

---



# 86. The Capacitated Facility Location Problem (CFLP)

## Tier 5: An Integrated Digital-Twin Simulation

### Purpose and scope

This notebook builds a small, synthetic digital-twin-style simulation around a capacitated facility allocation problem. It represents six facilities, eight customers, and the transportation links between every facility-customer pair. The simulation updates stylized demand and operating conditions hourly, assigns each customer to an available facility, records performance measures, and compares several what-if scenarios.

The allocation rule is a **demand-prioritized greedy heuristic**, not an exact optimizer: customers are considered from highest to lowest demand, and each is assigned to the feasible active facility with the lowest current cost per unit. Therefore, a result is a feasible allocation when all demand is served, but it is not guaranteed to minimize the total cost globally. The notebook uses the term *allocation heuristic* for this rule; the method is not a mathematical-programming solver.

This operational allocation layer is related to CFLP, but it does not make facility-opening decisions. All candidate facilities are treated as already available, and the facilities' fixed costs are stored as data but are not included in the allocation calculation. The model also omits partial customer assignments, inventory, production, facility-opening costs, calibrated sensor feeds, and real road-network routing. Its hourly updates are simulated data, not live synchronization with physical assets.

### Modeling assumptions

- Facility capacity is reduced in proportion to the facility's labor-efficiency factor.
- A customer is assigned wholly to one facility or left unassigned; unassigned demand incurs a penalty of $1,000 per demand unit.
- Transportation cost per demand unit combines distance-based base cost, a weather multiplier, a delay charge of $10 per delay-hour, and a facility-quality adjustment.
- Facility sensor utilization is generated by a stylized time pattern. It is not calculated from the allocation loads, so it should be interpreted as a simulated telemetry signal rather than a measured capacity-utilization rate.
- Demand volatility, sensor variation, delays, and weather effects use random draws. No random seed is set, so results can differ between runs.
- Scenario runs proceed in sequence in the same simulator and advance its clock. Scenario-specific changes are restored after each run, but the scenarios are not controlled experiments with identical random draws or identical starting timestamps.

### Allocation rule and cost model

For hour $t$, let $I$ be the set of facilities and $J$ the set of customers. Let $d_{jt}$ be customer $j$'s demand, $K_{it}$ the effective capacity of active facility $i$, and $c_{ijt}$ the cost per demand unit of serving customer $j$ from facility $i$. The code calculates the link cost as

$$
c_{ijt} = \left(b_{ij} w_{ijt} + 10\,\ell_{ijt}\right)
          \left(1 + 0.2(1-q_{it})\right),
$$

where $b_{ij}$ is the distance-based base cost per unit, $w_{ijt}$ is the weather multiplier, $\ell_{ijt}$ is delay in hours, and $q_{it}$ is the facility quality score. Effective capacity is $K_{it}=\text{capacity}_i\times\text{labor efficiency}_{it}$. The values are computed in `calculate_optimal_assignment`; despite that preserved function name, the procedure is a greedy heuristic rather than an exact optimization.

Customers are sorted in descending order of demand. For each customer in that order, the heuristic chooses the active facility with enough remaining effective capacity and the smallest $c_{ijt}$. If no such facility exists, the customer is unassigned and its demand incurs the penalty. This order matters: an early assignment can use capacity that a later customer might otherwise need. Consequently, locally cheapest feasible choices do not in general imply a globally minimum-cost allocation.

### Simulation and interpretation

Each hourly snapshot contains the simulated state, allocation cost, demand-weighted service level, system-efficiency indicator, and generated alerts. The notebook then compares baseline operations with a demand surge, a facility disruption, and a transportation-cost increase. Its short-horizon forecast extrapolates a linear trend from recent snapshots; the displayed uncertainty band is illustrative, not a statistically calibrated confidence interval. Use the outputs to explore how the stated heuristic and assumptions behave, not as a validated operational recommendation.

In [ ]:
from copy import deepcopy
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class Facility:
    """Facility attributes and simulated operational telemetry."""

    id: int
    name: str
    x_coord: float
    y_coord: float
    fixed_cost: float
    capacity: float
    current_utilization: float = 0.0
    operating_status: str = "ACTIVE"
    energy_consumption: float = 0.0
    labor_efficiency: float = 1.0
    quality_score: float = 1.0
    maintenance_schedule: Optional[List[str]] = None

    def __post_init__(self) -> None:
        if self.maintenance_schedule is None:
            self.maintenance_schedule = []


@dataclass
class Customer:
    """Customer location, demand, and service attributes."""

    id: int
    name: str
    x_coord: float
    y_coord: float
    base_demand: float
    demand_pattern: str = "STABLE"
    service_level_requirement: float = 0.95


@dataclass
class TransportationLink:
    """Transportation cost and simulated link conditions."""

    from_facility: int
    to_customer: int
    base_cost_per_unit: float
    current_delay: float = 0.0
    capacity_utilization: float = 0.0
    weather_impact: float = 1.0


@dataclass
class DigitalTwinState:
    """Independent snapshot of the simulated system at one timestamp."""

    timestamp: datetime
    facilities: List[Facility]
    customers: List[Customer]
    transportation_links: List[TransportationLink]
    total_cost: float
    service_level: float
    system_efficiency: float
    alerts: List[str]

In [ ]:
def initialize_facilities() -> List[Facility]:
    """Create candidate facilities and initialize synthetic telemetry."""
    facilities = [
        Facility(1, "North Hub", 20, 80, 150000, 1200),
        Facility(2, "Central DC", 50, 50, 120000, 1000),
        Facility(3, "South DC", 80, 20, 100000, 800),
        Facility(4, "East Facility", 85, 65, 110000, 900),
        Facility(5, "West Facility", 15, 35, 90000, 700),
        Facility(6, "Northeast DC", 70, 85, 130000, 1100),
    ]

    for facility in facilities:
        facility.current_utilization = np.random.uniform(0.3, 0.8)
        facility.energy_consumption = np.random.uniform(80, 95)
        facility.labor_efficiency = np.random.uniform(0.85, 1.0)
        facility.quality_score = np.random.uniform(0.9, 1.0)

    return facilities


def initialize_customers() -> List[Customer]:
    """Create customers with stable, seasonal, or volatile demand."""
    return [
        Customer(1, "Store A", 25, 75, 300, "STABLE"),
        Customer(2, "Store B", 45, 65, 450, "SEASONAL"),
        Customer(3, "Store C", 75, 35, 320, "VOLATILE"),
        Customer(4, "Store D", 90, 60, 420, "STABLE"),
        Customer(5, "Store E", 30, 20, 380, "SEASONAL"),
        Customer(6, "Store F", 60, 90, 460, "VOLATILE"),
        Customer(7, "Store G", 85, 40, 290, "STABLE"),
        Customer(8, "Store H", 55, 75, 410, "SEASONAL"),
    ]


def initialize_transportation_links(
    facilities: List[Facility], customers: List[Customer]
) -> List[TransportationLink]:
    """Create a link for every facility-customer pair."""
    links = []

    for facility in facilities:
        for customer in customers:
            distance = np.hypot(
                facility.x_coord - customer.x_coord,
                facility.y_coord - customer.y_coord,
            )
            base_cost = distance * 8.5
            link = TransportationLink(
                from_facility=facility.id,
                to_customer=customer.id,
                base_cost_per_unit=base_cost,
                current_delay=np.random.uniform(0, 2),
                capacity_utilization=np.random.uniform(0.3, 0.7),
                weather_impact=np.random.uniform(0.9, 1.1),
            )
            links.append(link)

    return links


facilities = initialize_facilities()
customers = initialize_customers()
transportation_links = initialize_transportation_links(facilities, customers)

data_point_count = (
    len(facilities) * 5
    + len(customers) * 2
    + len(transportation_links) * 4
)

print("Synthetic simulation initialized:")
print(f"Facilities: {len(facilities)}")
print(f"Customers: {len(customers)}")
print(f"Transportation links: {len(transportation_links)}")
print(f"Counted telemetry and attribute values: {data_point_count}")

In [ ]:
class DigitalTwinSimulator:
    """Hourly simulator for the capacitated facility allocation example."""

    def __init__(
        self,
        facilities: List[Facility],
        customers: List[Customer],
        transportation_links: List[TransportationLink],
    ) -> None:
        self.facilities = facilities
        self.customers = customers
        self.transportation_links = transportation_links
        self.current_time = datetime(2024, 1, 1, 0, 0, 0)
        self.history: List[DigitalTwinState] = []
        self.scenarios: List[Dict] = []

    def calculate_dynamic_demand(self, customer: Customer, hour: int) -> float:
        """Apply time-of-day and customer-specific demand variation."""
        time_factors = {
            "morning_peak": 1.3,
            "lunch_peak": 1.5,
            "evening_peak": 1.4,
            "off_peak": 0.7,
        }
        if 6 <= hour <= 10:
            time_factor = time_factors["morning_peak"]
        elif 11 <= hour <= 14:
            time_factor = time_factors["lunch_peak"]
        elif 17 <= hour <= 20:
            time_factor = time_factors["evening_peak"]
        else:
            time_factor = time_factors["off_peak"]

        if customer.demand_pattern == "SEASONAL":
            seasonal_factor = 1.0 + 0.2 * np.sin(2 * np.pi * hour / 24)
            time_factor *= seasonal_factor
        elif customer.demand_pattern == "VOLATILE":
            volatility = np.random.normal(1.0, 0.2)
            time_factor *= max(0.5, volatility)

        return customer.base_demand * time_factor

    def update_facility_sensors(self, hour: int) -> None:
        """Generate stylized facility telemetry for the current hour."""
        for facility in self.facilities:
            if facility.operating_status != "ACTIVE":
                continue

            target_utilization = 0.5 + 0.3 * np.sin(
                2 * np.pi * (hour - 6) / 24
            )
            facility.current_utilization += 0.1 * (
                target_utilization - facility.current_utilization
            )
            facility.current_utilization = np.clip(
                facility.current_utilization, 0, 1
            )

            facility.energy_consumption = (
                70
                + 25 * facility.current_utilization
                + np.random.normal(0, 2)
            )

            if 8 <= hour <= 17:
                facility.labor_efficiency = 0.95 + np.random.normal(0, 0.05)
            else:
                facility.labor_efficiency = 0.85 + np.random.normal(0, 0.08)
            facility.labor_efficiency = np.clip(
                facility.labor_efficiency, 0.7, 1.0
            )

            facility.quality_score = np.clip(
                facility.quality_score * 0.999, 0.8, 1.0
            )

    def update_transportation_conditions(self, hour: int) -> None:
        """Generate traffic, link-utilization, and weather conditions."""
        for link in self.transportation_links:
            if 7 <= hour <= 9 or 16 <= hour <= 18:
                link.current_delay = np.random.uniform(1, 3)
            else:
                link.current_delay = np.random.uniform(0, 1)

            link.capacity_utilization = 0.4 + 0.3 * np.sin(
                2 * np.pi * (hour - 8) / 24
            )
            link.capacity_utilization = np.clip(
                link.capacity_utilization, 0, 1
            )

            if 6 <= hour <= 18:
                link.weather_impact = 1.0 + 0.05 * np.random.normal(0, 1)
            else:
                link.weather_impact = 1.0 + 0.02 * np.random.normal(0, 1)
            link.weather_impact = np.clip(link.weather_impact, 0.8, 1.2)

    def calculate_optimal_assignment(self, hour: int) -> Dict:
        """Apply the demand-prioritized greedy allocation heuristic."""
        current_demands = {
            customer.id: self.calculate_dynamic_demand(customer, hour)
            for customer in self.customers
        }
        active_facilities = [
            facility
            for facility in self.facilities
            if facility.operating_status == "ACTIVE"
        ]
        facility_loads = {facility.id: 0.0 for facility in active_facilities}
        assignments: Dict[int, Optional[int]] = {}
        total_cost = 0.0

        sorted_customers = sorted(
            self.customers,
            key=lambda customer: current_demands[customer.id],
            reverse=True,
        )

        for customer in sorted_customers:
            demand = current_demands[customer.id]
            best_facility = None
            best_cost = float("inf")

            for facility in active_facilities:
                effective_capacity = (
                    facility.capacity * facility.labor_efficiency
                )
                if facility_loads[facility.id] + demand > effective_capacity:
                    continue

                link = next(
                    link
                    for link in self.transportation_links
                    if link.from_facility == facility.id
                    and link.to_customer == customer.id
                )
                transport_cost = (
                    link.base_cost_per_unit * link.weather_impact
                    + link.current_delay * 10
                )
                quality_adjustment = 1.0 + (
                    1.0 - facility.quality_score
                ) * 0.2
                cost_per_unit = transport_cost * quality_adjustment

                if cost_per_unit < best_cost:
                    best_cost = cost_per_unit
                    best_facility = facility.id

            if best_facility is not None:
                assignments[customer.id] = best_facility
                facility_loads[best_facility] += demand
                total_cost += best_cost * demand
            else:
                assignments[customer.id] = None
                total_cost += demand * 1000

        total_demand = sum(current_demands.values())
        unmet_demand = sum(
            current_demands[customer.id]
            for customer in self.customers
            if assignments[customer.id] is None
        )

        return {
            "assignments": assignments,
            "facility_loads": facility_loads,
            "total_cost": total_cost,
            "total_demand": total_demand,
            "unmet_demand": unmet_demand,
        }

    def calculate_system_metrics(self, assignment_result: Dict) -> Dict:
        """Calculate service, efficiency, and alert indicators."""
        total_demand = assignment_result["total_demand"]
        unmet_demand = assignment_result["unmet_demand"]
        service_level = (
            (total_demand - unmet_demand) / total_demand
            if total_demand > 0
            else 0.0
        )

        active_facilities = [
            facility
            for facility in self.facilities
            if facility.operating_status == "ACTIVE"
        ]
        if active_facilities:
            efficiency = sum(
                facility.current_utilization
                * facility.quality_score
                * facility.labor_efficiency
                for facility in active_facilities
            ) / len(active_facilities)
        else:
            efficiency = 0.0

        alerts = []
        for facility in self.facilities:
            if facility.current_utilization > 0.9:
                alerts.append(
                    f"HIGH_UTILIZATION: {facility.name} "
                    f"at {facility.current_utilization:.1%}"
                )
            if facility.quality_score < 0.85:
                alerts.append(
                    f"LOW_QUALITY: {facility.name} "
                    f"at {facility.quality_score:.2f}"
                )
        if service_level < 0.9:
            alerts.append(f"LOW_SERVICE_LEVEL: {service_level:.1%}")

        return {
            "service_level": service_level,
            "system_efficiency": efficiency,
            "alerts": alerts,
        }

    def advance_time(self, minutes: int = 60) -> DigitalTwinState:
        """Advance the clock, update simulated conditions, and record a snapshot."""
        self.current_time += timedelta(minutes=minutes)
        hour = self.current_time.hour

        self.update_facility_sensors(hour)
        self.update_transportation_conditions(hour)
        assignment_result = self.calculate_optimal_assignment(hour)
        metrics = self.calculate_system_metrics(assignment_result)

        state = DigitalTwinState(
            timestamp=self.current_time,
            facilities=deepcopy(self.facilities),
            customers=deepcopy(self.customers),
            transportation_links=deepcopy(self.transportation_links),
            total_cost=assignment_result["total_cost"],
            service_level=metrics["service_level"],
            system_efficiency=metrics["system_efficiency"],
            alerts=metrics["alerts"].copy(),
        )
        self.history.append(state)
        return state

    def run_scenario(
        self, scenario_name: str, duration_hours: int = 24
    ) -> List[DigitalTwinState]:
        """Run one scenario and restore its modified inputs afterward."""
        valid_scenarios = {
            "DEMAND_SURGE",
            "FACILITY_DISRUPTION",
            "COST_INCREASE",
        }
        if scenario_name not in valid_scenarios:
            raise ValueError(f"Unknown scenario: {scenario_name}")
        if duration_hours < 0:
            raise ValueError("duration_hours must be non-negative")

        print(f"\nRunning scenario: {scenario_name}")
        print(f"Duration: {duration_hours} hours")

        original_facilities = deepcopy(self.facilities)
        original_customers = deepcopy(self.customers)
        original_links = deepcopy(self.transportation_links)

        if scenario_name == "DEMAND_SURGE":
            for customer in self.customers:
                customer.base_demand *= 1.5
        elif scenario_name == "FACILITY_DISRUPTION":
            disrupted_facility = self.facilities[2]
            disrupted_facility.operating_status = "DISRUPTED"
            print(f"Disrupted facility: {disrupted_facility.name}")
        elif scenario_name == "COST_INCREASE":
            for link in self.transportation_links:
                link.base_cost_per_unit *= 1.3

        scenario_history = []
        for elapsed_hour in range(1, duration_hours + 1):
            state = self.advance_time(60)
            scenario_history.append(state)

            if (elapsed_hour - 1) % 6 == 0:
                print(
                    f"  Elapsed hour {elapsed_hour:2d}: "
                    f"Cost=${state.total_cost:,.0f}, "
                    f"Service={state.service_level:.1%}, "
                    f"Efficiency={state.system_efficiency:.1%}"
                )

        self.facilities = original_facilities
        self.customers = original_customers
        self.transportation_links = original_links
        self.scenarios.append(
            {"name": scenario_name, "history": scenario_history}
        )
        return scenario_history


print("Digital Twin Simulator class defined.")

In [ ]:
print("=" * 80)
print("BASELINE SIMULATION")
print("=" * 80)

simulator = DigitalTwinSimulator(
    facilities, customers, transportation_links
)
baseline_states = []

for elapsed_hour in range(1, 25):
    state = simulator.advance_time(60)
    baseline_states.append(state)

    if (elapsed_hour - 1) % 6 == 0:
        print(
            f"Elapsed hour {elapsed_hour:2d}: "
            f"${state.total_cost:,.0f}, "
            f"Service={state.service_level:.1%}, "
            f"Efficiency={state.system_efficiency:.1%}, "
            f"Alerts={len(state.alerts)}"
        )

avg_cost = np.mean([state.total_cost for state in baseline_states])
avg_service = np.mean([state.service_level for state in baseline_states])
avg_efficiency = np.mean(
    [state.system_efficiency for state in baseline_states]
)
total_baseline_alerts = sum(
    len(state.alerts) for state in baseline_states
)

print("\nBaseline summary:")
print(f"Average hourly cost: ${avg_cost:,.2f}")
print(f"Average service level: {avg_service:.1%}")
print(f"Average system-efficiency indicator: {avg_efficiency:.1%}")
print(f"Total generated alerts: {total_baseline_alerts}")

In [ ]:
print("\n" + "=" * 80)
print("WHAT-IF SCENARIO ANALYSIS")
print("=" * 80)

demand_surge_history = simulator.run_scenario("DEMAND_SURGE", 24)
facility_disruption_history = simulator.run_scenario(
    "FACILITY_DISRUPTION", 24
)
cost_increase_history = simulator.run_scenario("COST_INCREASE", 24)

scenario_histories = {
    "Baseline": baseline_states,
    "Demand Surge": demand_surge_history,
    "Facility Disruption": facility_disruption_history,
    "Cost Increase": cost_increase_history,
}

scenarios_data = []
for scenario_name, history in scenario_histories.items():
    scenarios_data.append(
        {
            "Scenario": scenario_name,
            "Avg Cost": np.mean([state.total_cost for state in history]),
            "Avg Service Level": np.mean(
                [state.service_level for state in history]
            ),
            "Avg Efficiency": np.mean(
                [state.system_efficiency for state in history]
            ),
            "Total Alerts": sum(len(state.alerts) for state in history),
        }
    )

comparison_df = pd.DataFrame(scenarios_data)
print("\nScenario summary:")
print(comparison_df.round(2).to_string(index=False))

baseline = scenarios_data[0]
print("\nPercentage changes relative to baseline:")
for scenario in scenarios_data[1:]:
    cost_change = (
        (scenario["Avg Cost"] / baseline["Avg Cost"] - 1) * 100
        if baseline["Avg Cost"] != 0
        else float("nan")
    )
    service_change = (
        (scenario["Avg Service Level"] / baseline["Avg Service Level"] - 1)
        * 100
        if baseline["Avg Service Level"] != 0
        else float("nan")
    )
    efficiency_change = (
        (scenario["Avg Efficiency"] / baseline["Avg Efficiency"] - 1) * 100
        if baseline["Avg Efficiency"] != 0
        else float("nan")
    )
    print(f"\n{scenario['Scenario']}:")
    print(f"  Average cost change: {cost_change:+.1f}%")
    print(f"  Average service-level change: {service_change:+.1f}%")
    print(f"  Average efficiency-indicator change: {efficiency_change:+.1f}%")

In [ ]:
def analyze_historical_patterns(
    history: List[DigitalTwinState],
) -> Dict:
    """Summarize observed hourly patterns and variation."""
    if not history:
        raise ValueError("history must contain at least one state")

    hours = [state.timestamp.hour for state in history]
    costs = [state.total_cost for state in history]
    service_levels = [state.service_level for state in history]

    hourly_stats = {}
    for hour in range(24):
        hour_states = [
            state
            for state, observed_hour in zip(history, hours)
            if observed_hour == hour
        ]
        if hour_states:
            hourly_stats[hour] = {
                "avg_cost": np.mean(
                    [state.total_cost for state in hour_states]
                ),
                "avg_service": np.mean(
                    [state.service_level for state in hour_states]
                ),
                "avg_efficiency": np.mean(
                    [state.system_efficiency for state in hour_states]
                ),
                "alert_count": sum(
                    len(state.alerts) for state in hour_states
                ),
            }

    peak_cost_hours = sorted(
        hourly_stats.items(),
        key=lambda item: item[1]["avg_cost"],
        reverse=True,
    )[:3]
    peak_alert_hours = sorted(
        hourly_stats.items(),
        key=lambda item: item[1]["alert_count"],
        reverse=True,
    )[:3]

    return {
        "hourly_stats": hourly_stats,
        "peak_cost_hours": peak_cost_hours,
        "peak_alert_hours": peak_alert_hours,
        "total_cost_variance": np.var(costs),
        "service_level_variance": np.var(service_levels),
    }


def predict_future_performance(
    history: List[DigitalTwinState], forecast_hours: int = 12
) -> Dict:
    """Extrapolate recent linear trends as an illustrative forecast."""
    if forecast_hours < 0:
        raise ValueError("forecast_hours must be non-negative")

    recent_history = history[-6:] if len(history) >= 6 else history
    if len(recent_history) >= 2:
        time_indices = np.arange(len(recent_history))
        cost_trend = np.polyfit(
            time_indices,
            [state.total_cost for state in recent_history],
            1,
        )[0]
        service_trend = np.polyfit(
            time_indices,
            [state.service_level for state in recent_history],
            1,
        )[0]
        efficiency_trend = np.polyfit(
            time_indices,
            [state.system_efficiency for state in recent_history],
            1,
        )[0]
    else:
        cost_trend = 0.0
        service_trend = 0.0
        efficiency_trend = 0.0

    predictions = []
    if history:
        last_state = history[-1]
        for hour_ahead in range(1, forecast_hours + 1):
            predicted_cost = last_state.total_cost + cost_trend * hour_ahead
            predicted_service = (
                last_state.service_level + service_trend * hour_ahead
            )
            predicted_efficiency = (
                last_state.system_efficiency
                + efficiency_trend * hour_ahead
            )
            cost_uncertainty = abs(cost_trend) * hour_ahead * 0.5
            predictions.append(
                {
                    "hour_ahead": hour_ahead,
                    "predicted_cost": predicted_cost,
                    "cost_lower": predicted_cost - cost_uncertainty,
                    "cost_upper": predicted_cost + cost_uncertainty,
                    "predicted_service": np.clip(
                        predicted_service, 0, 1
                    ),
                    "predicted_efficiency": np.clip(
                        predicted_efficiency, 0, 1
                    ),
                }
            )

    return {
        "predictions": predictions,
        "trends": {
            "cost_trend": cost_trend,
            "service_trend": service_trend,
            "efficiency_trend": efficiency_trend,
        },
    }


patterns = analyze_historical_patterns(baseline_states)
print("Historical pattern analysis:")
print("\nHighest average-cost hours:")
for hour, stats in patterns["peak_cost_hours"]:
    print(f"  Hour {hour:02d}: ${stats['avg_cost']:,.0f}")

print("\nHours with the most generated alerts:")
for hour, stats in patterns["peak_alert_hours"]:
    print(f"  Hour {hour:02d}: {stats['alert_count']} alerts")

predictions = predict_future_performance(baseline_states, 12)
print("\nIllustrative linear forecast (first six hours):")
for prediction in predictions["predictions"][:6]:
    print(
        f"  Hour +{prediction['hour_ahead']:2d}: "
        f"Cost=${prediction['predicted_cost']:,.0f} "
        f"(illustrative band ${prediction['cost_lower']:,.0f}"
        f"-${prediction['cost_upper']:,.0f}), "
        f"Service={prediction['predicted_service']:.1%}, "
        f"Efficiency={prediction['predicted_efficiency']:.1%}"
    )

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Digital-Twin Simulation Analysis", fontsize=16, fontweight="bold")

# Baseline cost and scaled performance indicators share one display axis.
ax1 = axes[0, 0]
hours = list(range(1, len(baseline_states) + 1))
costs = [state.total_cost for state in baseline_states]
service_scaled = [state.service_level * 1000 for state in baseline_states]
efficiency_scaled = [
    state.system_efficiency * 500 for state in baseline_states
]
ax1.plot(hours, costs, "b-", linewidth=2, label="Cost ($)")
ax1.plot(
    hours, service_scaled, "g--", linewidth=1,
    label="Service level (value × 1,000)",
)
ax1.plot(
    hours, efficiency_scaled, "r:", linewidth=1,
    label="Efficiency indicator (value × 500)",
)
ax1.set_xlabel("Elapsed hour")
ax1.set_ylabel("Cost or scaled indicator value")
ax1.set_title("Baseline hourly profile")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Compare mean simulated hourly costs across the four runs.
ax2 = axes[0, 1]
scenario_names = [scenario["Scenario"] for scenario in scenarios_data]
average_costs = [scenario["Avg Cost"] for scenario in scenarios_data]
colors = ["steelblue", "orange", "red", "purple"]
bars = ax2.bar(scenario_names, average_costs, color=colors, alpha=0.7)
ax2.set_ylabel("Average hourly cost ($)")
ax2.set_title("Scenario cost comparison")
ax2.grid(True, axis="y", alpha=0.3)
plt.setp(ax2.get_xticklabels(), rotation=35, ha="right")
label_offset = max(average_costs) * 0.01 if average_costs else 0
for bar, cost in zip(bars, average_costs):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + label_offset,
        f"${cost:,.0f}",
        ha="center",
        va="bottom",
    )

# Plot the simulated telemetry stored in each independent snapshot.
ax3 = axes[0, 2]
facility_names = [facility.name for facility in facilities]
for facility in facilities:
    utilization_history = [
        next(
            snapshot_facility.current_utilization
            for snapshot_facility in state.facilities
            if snapshot_facility.id == facility.id
        )
        for state in baseline_states
    ]
    ax3.plot(
        hours,
        utilization_history,
        marker="o",
        linewidth=1.5,
        markersize=3,
        label=facility.name,
    )
ax3.set_xlabel("Elapsed hour")
ax3.set_ylabel("Simulated utilization signal (fraction)")
ax3.set_title("Facility telemetry signals")
ax3.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
ax3.grid(True, alpha=0.3)

# Count generated alerts in each baseline snapshot.
ax4 = axes[1, 0]
alert_counts = [len(state.alerts) for state in baseline_states]
ax4.bar(hours, alert_counts, color="coral", alpha=0.7)
ax4.set_xlabel("Elapsed hour")
ax4.set_ylabel("Number of alerts")
ax4.set_title("Generated alerts by hour")
ax4.grid(True, axis="y", alpha=0.3)

# The shaded region is an illustrative trend-based band, not a confidence interval.
ax5 = axes[1, 1]
if predictions["predictions"]:
    forecast_hours = [
        prediction["hour_ahead"]
        for prediction in predictions["predictions"]
    ]
    forecast_costs = [
        prediction["predicted_cost"]
        for prediction in predictions["predictions"]
    ]
    cost_lower = [
        prediction["cost_lower"]
        for prediction in predictions["predictions"]
    ]
    cost_upper = [
        prediction["cost_upper"]
        for prediction in predictions["predictions"]
    ]
    ax5.plot(
        forecast_hours, forecast_costs, "b-", linewidth=2,
        label="Trend extrapolation",
    )
    ax5.fill_between(
        forecast_hours,
        cost_lower,
        cost_upper,
        alpha=0.3,
        color="blue",
        label="Illustrative uncertainty band",
    )
    ax5.set_xlabel("Hours ahead")
    ax5.set_ylabel("Forecast cost ($)")
    ax5.set_title("Short-horizon cost extrapolation")
    ax5.legend()
    ax5.grid(True, alpha=0.3)
else:
    ax5.set_title("No forecast states available")

# Compare normalized indicators; these scores are descriptive, not decision weights.
ax6 = axes[1, 2]
categories = [
    "Cost\nscore",
    "Service\nlevel",
    "System\nefficiency",
    "Alert\nscore",
]
category_count = len(categories)
cost_values = [scenario["Avg Cost"] for scenario in scenarios_data]
alert_values = [scenario["Total Alerts"] for scenario in scenarios_data]
cost_range = max(cost_values) - min(cost_values)
maximum_alerts = max(alert_values)
scenario_scores = []
for scenario in scenarios_data:
    cost_score = (
        1 - (scenario["Avg Cost"] - min(cost_values)) / cost_range
        if cost_range > 0
        else 1.0
    )
    alert_score = (
        1 - scenario["Total Alerts"] / maximum_alerts
        if maximum_alerts > 0
        else 1.0
    )
    scenario_scores.append(
        [
            cost_score,
            scenario["Avg Service Level"],
            scenario["Avg Efficiency"],
            alert_score,
        ]
    )

angles = np.linspace(0, 2 * np.pi, category_count, endpoint=False).tolist()
closed_angles = angles + angles[:1]
for scenario_name, scores in zip(scenario_names, scenario_scores):
    closed_scores = scores + scores[:1]
    ax6.plot(closed_angles, closed_scores, "o-", linewidth=2, label=scenario_name)
    ax6.fill(closed_angles, closed_scores, alpha=0.1)
ax6.set_xticks(angles)
ax6.set_xticklabels(categories)
ax6.set_ylim(0, 1)
ax6.set_title("Descriptive normalized comparison")
ax6.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
ax6.grid(True)

fig.tight_layout()
plt.show()

In [ ]:
print("\n" + "=" * 80)
print("SIMULATION INSIGHTS")
print("=" * 80)

print("\nFacility telemetry summary:")
for facility in facilities:
    facility_states = [
        snapshot_facility
        for state in baseline_states
        for snapshot_facility in state.facilities
        if snapshot_facility.id == facility.id
    ]
    if not facility_states:
        continue

    avg_utilization = np.mean(
        [snapshot_facility.current_utilization for snapshot_facility in facility_states]
    )
    avg_quality = np.mean(
        [snapshot_facility.quality_score for snapshot_facility in facility_states]
    )
    avg_labor_efficiency = np.mean(
        [snapshot_facility.labor_efficiency for snapshot_facility in facility_states]
    )

    print(f"\n{facility.name}:")
    print(f"  Average simulated utilization signal: {avg_utilization:.1%}")
    print(f"  Average quality score: {avg_quality:.3f}")
    print(f"  Average labor-efficiency factor: {avg_labor_efficiency:.1%}")

    if avg_utilization > 0.85:
        print("  Indicator: review the capacity assumptions")
    elif avg_utilization < 0.4:
        print("  Indicator: review the facility's assumed role")
    if avg_quality < 0.9:
        print("  Indicator: review the assumed maintenance schedule")
    if avg_labor_efficiency < 0.9:
        print("  Indicator: review the labor-efficiency assumptions")

print("\nTransportation links with high delay in the final baseline snapshot:")
baseline_final_links = baseline_states[-1].transportation_links
high_delay_links = [
    link for link in baseline_final_links if link.current_delay > 1.5
]
if high_delay_links:
    for link in high_delay_links[:3]:
        facility_name = next(
            facility.name
            for facility in facilities
            if facility.id == link.from_facility
        )
        customer_name = next(
            customer.name
            for customer in customers
            if customer.id == link.to_customer
        )
        print(
            f"  {facility_name} to {customer_name}: "
            f"{link.current_delay:.1f} hours"
        )
else:
    print("  No links exceed the illustrative delay threshold.")

print("\nRecent linear trend indicators:")
trends = predictions["trends"]
print(
    "Cost trend: "
    f"{'increasing' if trends['cost_trend'] > 0 else 'decreasing' if trends['cost_trend'] < 0 else 'stable'}"
)
print(
    "Service-level trend: "
    f"{'improving' if trends['service_trend'] > 0 else 'declining' if trends['service_trend'] < 0 else 'stable'}"
)
print(
    "System-efficiency-indicator trend: "
    f"{'increasing' if trends['efficiency_trend'] > 0 else 'decreasing' if trends['efficiency_trend'] < 0 else 'stable'}"
)

print("\nInterpretation notes:")
print("- Telemetry indicators are synthetic and are not calibrated measurements.")
print("- The allocation rule is greedy and does not include facility-opening decisions.")
print("- Scenario averages include random variation and sequential simulation times.")
print("- Forecast values extrapolate recent trends and are not validated predictions.")
print("\nSimulation coverage:")
print(f"Facilities: {len(facilities)}")
print(f"Customers: {len(customers)}")
print(f"What-if scenarios: {len(scenarios_data) - 1}")
print(f"Baseline hourly snapshots: {len(baseline_states)}")
print(f"Baseline alerts: {total_baseline_alerts}")

## Why this tier is useful

A static allocation model describes decisions for a specified set of inputs. A digital-twin-style simulation adds a time-indexed view: demand, link conditions, and facility indicators can vary from one snapshot to the next. This makes it possible to inspect operational trajectories and explore what-if changes under explicit assumptions. In this notebook, the simulated updates are hourly and synthetic; they do not constitute a live, synchronized digital twin.

### Strengths and limitations

**Potential strengths of this modeling approach**
- Brings facility, customer, and transportation attributes into one simulation state.
- Records time-indexed costs, service indicators, efficiency indicators, and alerts.
- Supports scenario exploration without requiring physical changes to a network.
- Makes assumptions and their operational consequences inspectable.

**Important limitations**
- Real-time use would require validated data pipelines, sensor calibration, and synchronization with operational systems.
- The allocation heuristic is not guaranteed to find a globally minimum-cost allocation.
- The model does not choose which facilities to open and does not account for their fixed costs.
- The telemetry patterns, penalty value, delay charge, and forecast band are illustrative assumptions rather than empirically estimated quantities.
- Scenario outcomes are stochastic and sequential, so differences should not be attributed solely to the scenario change.
- A deployed system would also need to address model validation, data quality, security, and operational governance.

### Appropriate use

This type of simulation can support learning, preliminary scenario discussions, and exploration of how assumptions interact in a multi-facility network. Applying it to operational decisions would require validated data, a clear treatment of facility-opening decisions and fixed costs, calibration against observed behavior, and a careful assessment of the allocation rule's limitations. Those extensions are outside the scope of this notebook.